# Chapitre 2 — Le DataFrame : construction et anatomie

## Le problème

Une Series décrit une seule variable. Une commande, un client ou un athlète se décrit par plusieurs variables de types différents. Le **DataFrame** réunit plusieurs Series qui partagent le même index : une table dont chaque colonne a son propre type.

Ce chapitre montre comment construire un DataFrame à partir des structures Python courantes, comment lire ses attributs, et comment se comportent les opérations quand deux tables ne sont pas alignées.

In [38]:
import numpy as np
import pandas as pd

## 1. Quatre façons de construire un DataFrame

### a) Un dictionnaire de Series

Chaque clé devient un nom de colonne. Les Series sont **alignées sur leur index**.

In [39]:
quantite = pd.Series([4, 4, 0, 4, 2], index=list("abcde"), dtype=np.uint8)
canal = pd.Series(["web", "magasin", "web", "telephone", "web"], index=list("abcde"), dtype="category")
remise = pd.Series([0.1, 0.0, 0.05], index=list("abd"))      # seulement 3 étiquettes

df = pd.DataFrame({"quantite": quantite, "canal": canal, "remise": remise})
df

,quantite,canal,remise
a,4,web,0.10
b,4,magasin,0.00
c,0,web,NaN
d,4,telephone,0.05
e,2,web,NaN


La Series `remise` n'a pas d'étiquette `c` ni `e` : ces cellules valent `NaN`. Chaque colonne garde le type de la Series d'origine.

In [40]:
df.dtypes

quantite       uint8
canal       category
remise       float64
dtype: object

### b) Un dictionnaire de listes

C'est la forme la plus courante pour créer un petit jeu de données à la main. Toutes les listes doivent avoir la même longueur.

In [41]:
produits = pd.DataFrame({
    "produit_id": ["P01", "P02", "P03"],
    "libelle": ["Clavier mécanique", "Souris sans fil", "Écran 27 pouces"],
    "prix": [89.9, 24.5, 249.0],
})
produits

,produit_id,libelle,prix
0,P01,Clavier mécanique,89.9
1,P02,Souris sans fil,24.5
2,P03,Écran 27 pouces,249.0


### c) Un tableau NumPy et une liste de noms de colonnes

Utile quand les données viennent d'un calcul numérique.

In [42]:
rng = np.random.default_rng(0)
print(rng)
mesures = pd.DataFrame(rng.normal(20, 2, size=(4, 3)).round(1), columns=["capteur_1", "capteur_2", "capteur_3"])
mesures

Generator(PCG64)


,capteur_1,capteur_2,capteur_3
0,20.3,19.7,21.3
1,20.2,18.9,20.7
2,22.6,21.9,18.6
3,17.5,18.8,20.1


### d) Une liste de dictionnaires

C'est la forme typique d'une réponse d'API en JSON : une liste d'enregistrements. Les clés absentes d'un enregistrement deviennent `NaN`.

In [43]:
enregistrements = [
    {"client_id": 101, "ville": "Anvers", "age": 40},
    {"client_id": 102, "ville": "Liège"},
    {"client_id": 103, "ville": "Charleroi", "age": 51, "segment": "Pro"},
]
pd.DataFrame(enregistrements)

,client_id,ville,age,segment
0,101,Anvers,40.0,NaN
1,102,Liège,NaN,NaN
2,103,Charleroi,51.0,Pro


> **Prédiction.** Quel sera le type de la colonne `age` dans le DataFrame ci-dessus ? Et celui de `segment` ? Vérifiez avec `.dtypes`.

In [44]:
pd.DataFrame(enregistrements).dtypes

client_id      int64
ville            str
age          float64
segment          str
dtype: object

## 2. Les types de données

Pandas reprend les types NumPy et en ajoute plusieurs :

| Type | Contenu | Remarque |
|---|---|---|
| `int64`, `int32`, `uint8`… | entiers | n'acceptent pas `NaN` |
| `float64`, `float32` | décimaux | acceptent `NaN` |
| `bool` | vrai / faux | |
| `str` | texte | type par défaut du texte en Pandas 3 |
| `object` | objets Python quelconques | ancien type du texte, encore utilisé pour les contenus mixtes |
| `category` | un nombre limité de modalités | économe en mémoire |
| `datetime64[ns]` | dates et heures | chapitre 5 |
| `Int64`, `Float64`, `boolean` | versions « nullables » | acceptent la valeur manquante `<NA>` |

Documentation : <https://pandas.pydata.org/docs/user_guide/basics.html#dtypes> et, pour les types de base, <https://numpy.org/doc/stable/user/basics.types.html>.

## 3. Les attributs d'un DataFrame

In [45]:
commandes = pd.read_csv("../donnees/demo/commandes.csv")
commandes.head()

,commande_id,client_id,produit_id,date_commande,quantite,canal,statut
0,5001,999,P08,2025-01-02 20:20:00,5,web,livrée
1,5002,105,P09,2025-01-04 19:56:00,1,web,livrée
2,5003,998,P12,2025-01-06 09:09:00,1,web,annulée
3,5004,121,P02,2025-01-06 18:49:00,1,magasin,annulée
4,5005,102,P11,2025-01-14 21:29:00,1,web,en cours


In [46]:
print("shape  :", commandes.shape)
print("index  :", commandes.index)
print("columns:", list(commandes.columns))

shape  : (240, 7)
index  : RangeIndex(start=0, stop=240, step=1)
columns: ['commande_id', 'client_id', 'produit_id', 'date_commande', 'quantite', 'canal', 'statut']


In [47]:
commandes.dtypes

commande_id      int64
client_id        int64
produit_id         str
date_commande      str
quantite         int64
canal              str
statut             str
dtype: object

`.to_numpy()` (ou l'ancien `.values`) renvoie un tableau NumPy à deux dimensions. Comme un tableau NumPy n'a qu'un seul type, les colonnes mélangées forcent le type `object` :

In [48]:
commandes.head(3).to_numpy()

array([[5001, 999, 'P08', '2025-01-02 20:20:00', 5, 'web', 'livrée'],
       [5002, 105, 'P09', '2025-01-04 19:56:00', 1, 'web', 'livrée'],
       [5003, 998, 'P12', '2025-01-06 09:09:00', 1, 'web', 'annulée']],
      dtype=object)

C'est pour cette raison qu'avant de passer des données à un modèle, on ne conserve que des colonnes numériques (chapitre 12).

## 4. Parcourir un DataFrame

`items()` parcourt les **colonnes** : on obtient le nom et la Series correspondante.

In [49]:
for nom, colonne in produits.items():
    print(f"{nom:10s} {colonne.dtype}  {colonne.tolist()}")

produit_id str  ['P01', 'P02', 'P03']
libelle    str  ['Clavier mécanique', 'Souris sans fil', 'Écran 27 pouces']
prix       float64  [89.9, 24.5, 249.0]


`iterrows()` parcourt les **lignes**, chacune sous forme de Series.

In [50]:
for i, ligne in produits.iterrows():
    print(i, ligne["libelle"], ligne["prix"])

0 Clavier mécanique 89.9
1 Souris sans fil 24.5
2 Écran 27 pouces 249.0


Ces boucles servent à afficher ou à déboguer. **Pour calculer, on ne parcourt pas les lignes** : on écrit une opération sur des colonnes entières, qui s'exécute en code compilé. Le chapitre 10 mesure l'écart : il dépasse souvent un facteur 100.

## 5. Opérations sur un DataFrame

### Avec un scalaire

In [51]:
mesures + 1

,capteur_1,capteur_2,capteur_3
0,21.3,20.7,22.3
1,21.2,19.9,21.7
2,23.6,22.9,19.6
3,18.5,19.8,21.1


> **Trouver l'erreur.** La cellule suivante échoue. Lisez le message d'erreur et identifiez la colonne responsable.

In [52]:
df + 1

TypeError: unsupported operand type(s) for +: 'Categorical' and 'int'

L'opération est appliquée à **toutes les colonnes**. Additionner 1 à une colonne `category` n'a pas de sens : `canal` est une variable qualitative. On sélectionne donc les colonnes concernées (chapitre 4).

In [53]:
df[["quantite", "remise"]] + 1

,quantite,remise
a,5,1.10
b,5,1.00
c,1,NaN
d,5,1.05
e,3,NaN


### Entre deux DataFrame : alignement sur les lignes **et** les colonnes

In [54]:
ventes_janvier = pd.DataFrame({"web": [10, 4], "magasin": [3, 8]}, index=["P01", "P02"])
ventes_fevrier = pd.DataFrame({"magasin": [5, 1], "telephone": [2, 0]}, index=["P02", "P03"])
ventes_janvier

,web,magasin
P01,10,3
P02,4,8


In [55]:
ventes_fevrier

,magasin,telephone
P02,5,2
P03,1,0


> **Prédiction.** Quelles dimensions aura `ventes_janvier + ventes_fevrier` ? Combien de cellules ne contiendront pas `NaN` ?

In [56]:
ventes_janvier + ventes_fevrier

,magasin,telephone,web
P01,NaN,NaN,NaN
P02,13.0,NaN,NaN
P03,NaN,NaN,NaN


Le résultat contient l'union des lignes (`P01`, `P02`, `P03`) et l'union des colonnes (`magasin`, `telephone`, `web`). Une seule cellule existe des deux côtés : `(P02, magasin)`. Avec `fill_value`, les cellules absentes d'un seul côté sont considérées comme 0 ; celles absentes des deux côtés restent `NaN`.

In [57]:
ventes_janvier.add(ventes_fevrier, fill_value=0)

,magasin,telephone,web
P01,3.0,NaN,10.0
P02,13.0,2.0,4.0
P03,1.0,0.0,NaN


### Fonctions NumPy

In [58]:
np.sqrt(mesures)

,capteur_1,capteur_2,capteur_3
0,4.505552,4.438468,4.615192
1,4.494441,4.347413,4.549725
2,4.753946,4.679744,4.312772
3,4.183300,4.335897,4.483302


### Division entière et reste

Les opérateurs `//` et `%` fonctionnent colonne par colonne. Exemple : convertir des durées en minutes au format heures et minutes.

In [68]:
durees = pd.DataFrame({"film": ["A", "B", "C"], "duree_min": [86, 117, 185]})
durees["heures"] = durees["duree_min"] // 60
durees["minutes"] = durees["duree_min"] % 60
durees

,film,duree_min,heures,minutes
0,A,86,1,26
1,B,117,1,57
2,C,185,3,5


## Pièges fréquents

| Situation | Ce qui se passe | Réflexe |
|---|---|---|
| Listes de longueurs différentes dans un dict | `ValueError: All arrays must be of the same length` | vérifier les longueurs, ou passer par des Series |
| Opération sur tout le DataFrame avec des colonnes texte ou catégorie | `TypeError` | sélectionner d'abord les colonnes numériques |
| Additionner deux tables mal alignées | des `NaN` partout | comparer `index` et `columns` des deux côtés |
| `to_numpy()` sur des colonnes mixtes | un tableau `object`, lent et inutilisable par les modèles | ne garder que les colonnes numériques |

## Exercices

- Drill **D07 Pokemon** (`exercices/drills/originaux/07_Creating_Series_and_DataFrames/Pokemon/`), traduction dans `exercices/drills/traductions_fr/`. Remarque : l'énoncé affirme que les colonnes apparaissent dans l'ordre alphabétique. Ce n'est plus vrai depuis Python 3.7 : les dictionnaires conservent l'ordre d'insertion. L'étape 4 reste un bon exercice de réordonnancement.

## À retenir

- Un DataFrame = des colonnes (Series) qui partagent un index.
- On le construit à partir d'un dict de Series, d'un dict de listes, d'un tableau NumPy, ou d'une liste de dicts.
- `shape`, `dtypes`, `columns`, `index` décrivent sa structure ; `to_numpy()` donne le tableau brut.
- Les opérations s'alignent sur les lignes **et** sur les colonnes.
- On parcourt les lignes pour inspecter, jamais pour calculer.